# 00 · Verificación del entorno

Este notebook no es de contenido del curso: es un chequeo rápido para
confirmar que tu entorno quedó bien configurado **antes de la primera
clase**.

Instrucciones:

1. Sigue la sección "Setup" del `README.md` (`uv sync`) si todavía no lo
   has hecho.
2. Abre este notebook con el kernel del curso — el que está dentro de la
   carpeta `.venv` del repo (ver "Abriendo los notebooks con el kernel
   correcto" en el README si no sabes cómo) — y ejecútalo completo:
   **Run All** / "Ejecutar todas las celdas".
3. Revisa la salida de cada celda. Lo que buscas es ✅ en todo.

Si alguna celda marca ❌, el mensaje casi siempre dice qué hacer. Si no
logras resolverlo, escríbele al profesor **antes de la clase** con el
texto completo del error (cópialo, no solo una captura recortada).

In [ ]:
# Ejecuta esta celda primero: solo define funciones para reportar el
# resultado de cada chequeo y no debería fallar nunca.
resultados = []


def ok(mensaje):
    print(f"✅ {mensaje}")
    resultados.append((mensaje, True))


def fail(mensaje, ayuda=""):
    print(f"❌ {mensaje}")
    if ayuda:
        print(f"   → {ayuda}")
    resultados.append((mensaje, False))

## 1. Python y entorno virtual

El curso usa `uv` para manejar el entorno. Si el kernel que estás usando
no es el `.venv` que crea `uv sync`, vas a tener paquetes faltantes aunque
la instalación haya salido bien.

In [ ]:
import sys

REQUISITO_PYTHON = (3, 12)

if sys.version_info[:2] >= REQUISITO_PYTHON:
    ok(f"Python {sys.version.split()[0]}")
else:
    fail(
        f"Python {sys.version.split()[0]} (se requiere {REQUISITO_PYTHON[0]}.{REQUISITO_PYTHON[1]}+)",
        "Instala una versión más nueva de Python y vuelve a correr `uv sync`.",
    )

if ".venv" in sys.executable:
    ok(f"Kernel del entorno virtual del curso ({sys.executable})")
else:
    fail(
        f"El kernel activo no parece ser el .venv de uv ({sys.executable})",
        "En VS Code / Jupyter, selecciona el kernel del entorno del proyecto "
        "(dentro de .venv), o corre `uv run jupyter lab` desde la carpeta del repo "
        "para que abra con el kernel correcto.",
    )

## 2. Librerías del curso

Confirma que `uv sync` instaló todo lo que se usa en las sesiones.

In [ ]:
import importlib
from importlib.metadata import PackageNotFoundError, version

LIBRERIAS = {
    "numpy": "numpy",
    "pandas": "pandas",
    "scikit-learn": "sklearn",
    "matplotlib": "matplotlib",
    "requests": "requests",
    "python-dotenv": "dotenv",
}

for paquete, modulo in LIBRERIAS.items():
    try:
        importlib.import_module(modulo)
        ok(f"{paquete} {version(paquete)}")
    except (ImportError, PackageNotFoundError):
        fail(f"No se pudo importar {paquete}", "Corre `uv sync` en la carpeta del repo.")

## 3. Conexión a internet y a los servicios del curso

Los notebooks de clase descargan sus datos en vivo, y los retos se
entregan contra un backend en AWS. Esta sección prueba ambas rutas de
red — típicamente lo que falla aquí es un firewall institucional, una
VPN, o wifi de universidad/empresa que bloquea tráfico saliente.

In [ ]:
import os

import requests

CLOUDFRONT_URL = "https://d3qixogk4zgixq.cloudfront.net"

try:
    resp = requests.head(CLOUDFRONT_URL, timeout=10)
    if resp.status_code < 500:
        ok("Conexión a los datos del curso (CloudFront)")
    else:
        fail(
            f"CloudFront respondió con status {resp.status_code}",
            "Puede ser temporal; intenta de nuevo en unos minutos.",
        )
except requests.RequestException as e:
    fail(
        "No se pudo conectar a los datos del curso (CloudFront)",
        f"Revisa tu conexión a internet, o si una red institucional/VPN está "
        f"bloqueando el tráfico saliente. Detalle técnico: {e}",
    )

In [ ]:
# La URL por defecto es la misma que aparece en el README. Si ya tienes tu
# .env configurado con otra, se usa esa.
API_URL = os.environ.get("ML_COURSE_API_URL") or "https://6trg8jthgl.execute-api.us-east-1.amazonaws.com/dev"

try:
    resp = requests.get(f"{API_URL}/assignments/tasacion-diamantes/leaderboard", timeout=10)
    if resp.status_code == 200:
        ok("Conexión al backend de entregas (API)")
    else:
        fail(
            f"La API respondió con status {resp.status_code}",
            "Puede ser temporal; intenta de nuevo en unos minutos.",
        )
except requests.RequestException as e:
    fail(
        "No se pudo conectar al backend de entregas (API)",
        f"Revisa tu conexión a internet, o si una red institucional/VPN está "
        f"bloqueando el tráfico saliente. Detalle técnico: {e}",
    )

## 4. Tu API key (opcional por ahora)

Tu API key la entrega el profesor — si todavía no la tienes, está bien:
solo la necesitas antes de entregar el primer reto, no para esta
verificación. Si ya la tienes y ya copiaste `.env.example` a `.env`, esta
celda confirma que quedó bien puesta.

In [ ]:
from pathlib import Path

from dotenv import load_dotenv

env_path = Path(".env")

if not env_path.exists():
    print("ℹ️  Todavía no existe un archivo .env — normal si aún no tienes tu API key.")
    print("   Cuando el profesor te la entregue: copia .env.example a .env y pégala ahí.")
else:
    load_dotenv()
    api_key = os.environ.get("ML_COURSE_API_KEY")
    if not api_key:
        fail("Existe .env pero ML_COURSE_API_KEY está vacío", "Pega tu API key en el archivo .env.")
    else:
        try:
            resp = requests.get(f"{API_URL}/submissions/me", headers={"x-api-key": api_key}, timeout=10)
            if resp.status_code == 200:
                ok("Tu API key es válida")
            elif resp.status_code == 401:
                fail(
                    "Tu API key no es válida",
                    "Revisa que la copiaste completa en .env, sin espacios ni comillas extra.",
                )
            else:
                fail(f"No se pudo validar la API key (status {resp.status_code})", "Intenta de nuevo más tarde.")
        except requests.RequestException as e:
            fail("No se pudo validar la API key por un problema de red", str(e))

## Resumen

In [ ]:
fallidos = [mensaje for mensaje, exito in resultados if not exito]

print("=" * 50)
if not fallidos:
    print(f"✅ Todo listo ({len(resultados)}/{len(resultados)} verificaciones OK). Nos vemos en clase.")
else:
    print(f"❌ {len(fallidos)} de {len(resultados)} verificaciones fallaron:")
    for mensaje in fallidos:
        print(f"   - {mensaje}")
    print()
    print("Revisa los mensajes de arriba. Si no logras resolverlo, escríbele al")
    print("profesor antes de la clase con el error completo.")